# Evaluation 04 — E0–E5: run registry, per-class results, real vs synthetic, paired comparisons

This notebook ingests existing final-test prediction files; it does **not** train a classifier or invent metrics. Label audit must have passed. Pairwise comparisons use **identical real-test IDs and group IDs**, within the same model and seed. No p-values are manufactured.

**Saved evidence:** `notebook_console.log` captures execution/tracebacks, `run_log.jsonl` records start and completed artifact SHA-256 checksums, and each analysis writes its reported CSV/JSON or graphic files into the configured Kaggle output directory. Keep per-image records and raw console text private; export only reviewed aggregate reports to GitHub.


In [ ]:
# Common Kaggle bootstrap: either attach DermSynReal source as Kaggle Input,
# or set CLONE_REPO_IF_MISSING=True and enable Kaggle Internet.
from pathlib import Path
import subprocess,sys,os,json
REPO_ROOT=Path("/kaggle/input/EDIT_REPO_DATASET/DermSynReal")
CLONE_REPO_IF_MISSING=True
EXPECTED_GIT_SHA=""  # Recommended: set a full pinned SHA for reproducible final runs.
if not (REPO_ROOT/"src/dermsynreal/evaluation.py").is_file():
    if not CLONE_REPO_IF_MISSING:
        raise FileNotFoundError("Attach the repository and change REPO_ROOT, or enable Git clone.")
    REPO_ROOT=Path("/kaggle/working/DermSynReal")
    if not (REPO_ROOT/".git").is_dir():
        subprocess.check_call(["git","clone","https://github.com/shanmukhdatta/DermSynReal.git",str(REPO_ROOT)])
if EXPECTED_GIT_SHA:
    if not (REPO_ROOT/".git").is_dir():
        raise ValueError("For pinned SHA verification attach a Git checkout or enable clone")
    actual=subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip()
    if actual!=EXPECTED_GIT_SHA:
        raise ValueError(f"Repository SHA {actual} differs from pinned SHA {EXPECTED_GIT_SHA}")
elif (REPO_ROOT/".git").is_dir():
    print("Current Git SHA:",subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip())
    print("WARNING: pin EXPECTED_GIT_SHA for final research outputs")
sys.path.insert(0,str(REPO_ROOT/"src"))
print("DermSynReal source:",REPO_ROOT)


## Run registry format
One row per final checkpoint. Required columns: `run_id,condition,backbone,seed,predictions_csv,checkpoint_sha,config_sha,real_split_sha`. See `reports/templates/runs_registry_example.csv` for header. Put all runs from one frozen real split into one registry.

In [ ]:
RUN_REGISTRY_CSV="/kaggle/input/EDIT_FINAL_RUNS/runs_registry.csv"
PASSED_LABEL_AUDIT_JSON="/kaggle/input/EDIT_REAL_AUDIT/label_audit_summary.json"
OUTPUT_DIR="/kaggle/working/dermsynreal_eval_04"
BOOTSTRAP_ITERATIONS=2000
BOOTSTRAP_SEED=42
RUN_FINAL_REPORT=False  # Only after all comparisons/selection are frozen.

In [ ]:
# Preserve a complete Kaggle notebook console and structured start event.
# This log can contain image file paths or case notes; keep it out of public GitHub.
from dermsynreal.kaggle_runs import start_notebook_logging,append_run_log
EVALUATION_OUTPUT=Path(OUTPUT_DIR)
PRIVATE_EVALUATION_LOG=start_notebook_logging(EVALUATION_OUTPUT,
    kind='evaluation_04',config={
        'notebook':'04_Experiment_Suite_Real_Test_Reporting.ipynb','data_verified':False,
        'run_enabled':bool(RUN_FINAL_REPORT)},repo_root=REPO_ROOT)
print('Saved notebook console:',PRIVATE_EVALUATION_LOG)


In [ ]:
from dermsynreal.evaluation_pipeline import run_reporting
if RUN_FINAL_REPORT:
    info=run_reporting(RUN_REGISTRY_CSV,OUTPUT_DIR,
        label_audit_json=PASSED_LABEL_AUDIT_JSON,iterations=BOOTSTRAP_ITERATIONS,
        bootstrap_seed=BOOTSTRAP_SEED)
    print(json.dumps(info,indent=2))
    import pandas as pd
    report=pd.read_csv(Path(OUTPUT_DIR)/"run_summary.csv")
    display(report)
    paired=pd.read_csv(Path(OUTPUT_DIR)/"paired_real_test.csv")
    display(paired)
else:
    print("Not executed. Provide a *passed* label audit and real checkpoint predictions first.")


In [ ]:
if RUN_FINAL_REPORT:
    import matplotlib.pyplot as plt
    view=report.sort_values(["backbone","seed","condition"])
    for backbone,group in view.groupby("backbone"):
        plt.figure(figsize=(10,4))
        plt.bar(group["run_id"].astype(str),group["real_test_macro_f1"])
        plt.ylim(0,1);plt.title(f"REAL-test macro-F1 — {backbone}")
        plt.ylabel("Macro-F1");plt.xticks(rotation=55,ha="right")
        plt.tight_layout()
        safe="".join(c if c.isalnum() or c in "-_" else "_" for c in backbone)
        plt.savefig(Path(OUTPUT_DIR)/f"macro_f1_real_{safe}.png",dpi=170)
        plt.show()


In [ ]:
# Confusion matrices: computed from actual saved row counts; never substitute mock results.
if RUN_FINAL_REPORT:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from dermsynreal.evaluation import CLASSES
    confusion_df=pd.read_csv(Path(OUTPUT_DIR)/"confusion_real_test.csv")
    for run_id,group in confusion_df.groupby("run_id"):
        matrix=group.pivot(index="true_label",columns="pred_label",values="count")
        matrix=matrix.reindex(index=CLASSES,columns=CLASSES,fill_value=0).fillna(0)
        fig,ax=plt.subplots(figsize=(8,6))
        im=ax.imshow(matrix.values)
        ax.set_xticks(range(6),CLASSES,rotation=50,ha="right")
        ax.set_yticks(range(6),CLASSES)
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title(f"Real test confusion matrix — {run_id}")
        for i in range(6):
            for j in range(6):
                ax.text(j,i,str(int(matrix.iloc[i,j])),ha="center",va="center")
        fig.colorbar(im,ax=ax)
        fig.tight_layout()
        safe="".join(c if c.isalnum() or c in "-_" else "_" for c in str(run_id))
        fig.savefig(Path(OUTPUT_DIR)/f"confusion_{safe}.png",dpi=170)
        plt.show()


The bootstrap uses protective groups; visually inferred groups do **not** guarantee patient-level independence. Report missing optional contrasts, multiple-testing decisions, and variability across seeds honestly.

In [ ]:
# Record checksums only for files actually created by this executed run.
if RUN_FINAL_REPORT:
    _files=[Path(OUTPUT_DIR)/'run_summary.csv',Path(OUTPUT_DIR)/'paired_real_test.csv',Path(OUTPUT_DIR)/'seed_summary_real_test.csv',Path(OUTPUT_DIR)/'report_manifest.json']
    _log=append_run_log(EVALUATION_OUTPUT,kind='evaluation_04_finished',
        config={'notebook':'04_Experiment_Suite_Real_Test_Reporting.ipynb','status':'completed_artifacts_written'},
        artifacts=[p for p in _files if p.is_file()],repo_root=REPO_ROOT)
    print('Evidence hashes:',_log['artifacts_sha256'])
